In [6]:
# Import Dependencies

import pandas as pd
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain.llms import OpenAI
import re
from dataclasses import dataclass
from typing import List, Optional, Dict

# ----------------------------
# 1) Parsing helpers
# ----------------------------

def parse_amount_pl(x: str) -> float:
    """Parse amounts like '-1 464,99' into -1464.99"""
    if pd.isna(x):
        return 0.0
    s = str(x).strip()
    s = s.replace("\u00a0", "").replace(" ", "")  # remove spaces & NBSP
    s = s.replace(",", ".")
    s = re.sub(r"[^0-9\.\-]", "", s)
    return float(s) if s else 0.0

def normalize_text(x: Optional[str]) -> str:
    if x is None or pd.isna(x):
        return ""
    return str(x).strip()

def make_month_col(dt: pd.Series) -> pd.Series:
    return dt.dt.to_period("M").astype(str)

# ----------------------------
# 2) Category mapping (dynamic rules)
# ----------------------------

@dataclass(frozen=True)
class CategoryRule:
    category: str
    pattern: str  # regex
    fields: Optional[List[str]] = None  # which columns to search in (default: common ones)

DEFAULT_FIELDS = ["counterparty", "title", "operation_type", "counterparty_address"]

def apply_category_rules(
    df: pd.DataFrame,
    rules,
    out_col: str = "category",
    default_category: str = "Uncategorized",
    text_col: str = "_search_text",
) -> pd.DataFrame:
    """
    Assign categories by regex rules. First match wins.
    Builds a helper column text_col by concatenating chosen fields.
    """
    df = df.copy()
    df[out_col] = default_category

    # Build searchable text once (cheap + avoids Series truth issues)
    # We create an empty Series aligned with df index.
    search = pd.Series([""] * len(df), index=df.index, dtype="string")

    # We'll build it per-rule (fields can differ), but most rules share DEFAULT_FIELDS.
    # To keep it simple and fast: build a base search text for DEFAULT_FIELDS once.
    def build_search_text(fields: List[str]) -> pd.Series:
        parts = []
        for f in fields:
            if f in df.columns:
                parts.append(df[f].fillna("").astype(str))
        if not parts:
            return pd.Series([""] * len(df), index=df.index, dtype="string")
        return parts[0].str.cat(parts[1:], sep=" ", na_rep="")

    # Cache for common field sets to avoid rebuilding
    cache = {}

    for rule in rules:
        fields = rule.fields or DEFAULT_FIELDS
        key = tuple(fields)

        if key not in cache:
            cache[key] = build_search_text(list(fields)).str.lower()

        searchable = cache[key]

        mask = searchable.str.contains(rule.pattern, regex=True, na=False)
        df.loc[mask & (df[out_col] == default_category), out_col] = rule.category

    return df
def list_top_uncategorized(
    df: pd.DataFrame,
    n_merchants: int = 20,
    min_total: float = 0.0,
) -> pd.DataFrame:
    """
    Helps you refine rules: shows biggest uncategorized merchants by spend.
    Expects: direction, abs_amount, counterparty, category.
    """
    tmp = df[(df["direction"] == "expense") & (df["category"] == "Uncategorized")].copy()
    if tmp.empty:
        return pd.DataFrame(columns=["counterparty", "total_spent", "tx_count"])

    out = (
        tmp.groupby("counterparty")
        .agg(total_spent=("abs_amount", "sum"), tx_count=("abs_amount", "size"))
        .reset_index()
        .sort_values("total_spent", ascending=False)
    )
    if min_total > 0:
        out = out[out["total_spent"] >= min_total]
    return out.head(n_merchants)

# ----------------------------
# 3) Monthly spend by category
# ----------------------------

def monthly_spend_by_category(df: pd.DataFrame) -> pd.DataFrame:
    """
    Returns a pivot table: month x category = total spent (positive).
    Expects: booking_date, direction, amount, category
    """
    expenses = df[df["direction"] == "expense"].copy()
    expenses["month"] = make_month_col(expenses["booking_date"])

    # spend as positive number
    expenses["spent"] = -expenses["amount"]

    pivot = (
        expenses.pivot_table(
            index="month",
            columns="category",
            values="spent",
            aggfunc="sum",
            fill_value=0.0,
        )
        .sort_index()
    )

    # add total column for convenience
    pivot["TOTAL"] = pivot.sum(axis=1)
    return pivot

def monthly_spend_long(df: pd.DataFrame) -> pd.DataFrame:
    """
    Long-format version: month, category, amount_spent
    """
    pivot = monthly_spend_by_category(df)
    pivot_no_total = pivot.drop(columns=["TOTAL"], errors="ignore")
    long = pivot_no_total.reset_index().melt(id_vars="month", var_name="category", value_name="amount_spent")
    long = long[long["amount_spent"] != 0].sort_values(["month", "amount_spent"], ascending=[True, False])
    return long

# ----------------------------
# 4) Loader for your Polish bank export
# ----------------------------

COLUMN_MAP = {
    "Data księgowania": "booking_date",
    "Data waluty": "value_date",
    "Nadawca / Odbiorca": "counterparty",
    "Adres nadawcy / odbiorcy": "counterparty_address",
    "Rachunek źródłowy": "source_account",
    "Rachunek docelowy": "target_account",
    "Tytułem": "title",
    "Kwota operacji": "amount",
    "Waluta": "currency",
    "Numer referencyjny": "reference",
    "Typ operacji": "operation_type",
}

def load_bank_export_csv(path: str) -> pd.DataFrame:
    """
    Robust-ish loader:
    - assumes ';' delimiter
    - keeps strings first, then parses dates/amount
    """
    df = pd.read_csv(path, sep=";", dtype=str)
    df = df.rename(columns=COLUMN_MAP)

    # dates
    df["booking_date"] = pd.to_datetime(df["booking_date"], format="%d.%m.%Y", errors="coerce")
    if "value_date" in df.columns:
        df["value_date"] = pd.to_datetime(df["value_date"], format="%d.%m.%Y", errors="coerce")

    # amount
    df["amount"] = df["amount"].apply(parse_amount_pl)

    # normalize text cols we use for categorization
    for col in ["counterparty", "title", "operation_type", "counterparty_address"]:
        if col in df.columns:
            df[col] = df[col].apply(normalize_text)

    # direction + abs
    df["direction"] = df["amount"].apply(lambda v: "expense" if v < 0 else "income")
    df["abs_amount"] = df["amount"].abs()

    return df

# ----------------------------
# 5) Example rules (edit freely)
# ----------------------------

BASE_RULES = [
    CategoryRule("Accounting", r"\b(wfirma|faktura|invoice|ksieg|księg|autopay|innovative)\b"),
    CategoryRule("Intermediary", r"\b(blik)\b"),
    CategoryRule("Groceries", r"\b(biedronka|lidl|zabka|żabka|kaufland|auchan|carrefour)\b"),
    CategoryRule("Transport", r"\b(uber|bolt|jakdojade|mpk|zkm)\b"),
    CategoryRule("Food", r"\b(pyszne|glovo|wolt|kfc|mcdonald|pizza|sushi|restauracja|wok|lisek)\b"),
    CategoryRule("Cofee", r"\b(kawiarnia)\b"),
    CategoryRule("Subscriptions", r"\b(netflix|spotify|google|apple|microsoft|adobe|jdmi)\b"),
    CategoryRule("Services", r"\b(barber|implant|ares)\b"),
    CategoryRule("Apartment", r"\b(agnieszka)\b"),
]




In [4]:
# Load and Preprocess CSV

csv_file_path = 'oct2025-march2026.csv'
df = load_bank_export_csv(csv_file_path)

# 2) Categorize (dynamic: just extend BASE_RULES)
df = apply_category_rules(df, BASE_RULES)

# 3) Monthly spend per category (pivot)
pivot = monthly_spend_by_category(df)
print("\nMonth x Category spending (PLN, positive):")
print(pivot.round(2))

# 4) Optional: long format (nice for charts / BI)
long = monthly_spend_long(df)
print("\nLong format (month, category, amount_spent):")
print(long.head(30))

# 5) Optional: show biggest uncategorized to refine rules
todo = list_top_uncategorized(df, n_merchants=25, min_total=100.0)
print("\nTop uncategorized merchants (>= 100 PLN total):")
print(todo)


Month x Category spending (PLN, positive):
category  Accounting  Apartment  Cofee    Food  Groceries  Intermediary  \
month                                                                     
2025-11      2822.93       3000   55.9  520.82     469.10       1437.00   
2025-12      2982.93       3000   86.5  538.30    1130.61       1002.44   
2026-01      2722.43          0   69.2  218.53    1183.91        745.52   
2026-02      2386.19       3250   51.9    0.00     545.62        180.94   

category  Services  Subscriptions  Transport  Uncategorized     TOTAL  
month                                                                  
2025-11          0            0.0      97.14         499.93   8902.82  
2025-12        720           62.9     140.77        3205.56  12870.01  
2026-01        740           62.9     119.10        3768.91   9630.50  
2026-02        355           62.9      25.13         555.68   7413.36  

Long format (month, category, amount_spent):
      month       category 

/var/folders/hm/m8h6dpgn1_gbp0z7xgp0gld00000gn/T/ipykernel_39915/1692158525.py:81: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = searchable.str.contains(rule.pattern, regex=True, na=False)


In [5]:
import pandas as pd

# Assumes df has at least:
# - booking_date (datetime)
# - amount (float, negative for expenses, positive for income)
# - direction ("expense"/"income")
# - currency (e.g., "PLN")
# - category (string)  # if you don't have it yet, create it earlier

currency = df["currency"].dropna().mode().iloc[0] if "currency" in df.columns and not df["currency"].dropna().empty else "PLN"

# --- Expenses only (recommended baseline for “spending”) ---
expenses = df[df["direction"] == "expense"].copy()

# Total spending (as positive number)
total_spending = expenses["amount"].sum()          # negative
total_spending_abs = -total_spending              # positive

print(f"\nTotal Spending: {total_spending_abs:.2f} {currency}")

# Unique categories (handle missing category column safely)
if "category" in expenses.columns:
    unique_categories_count = expenses["category"].nunique(dropna=True)
else:
    unique_categories_count = 0
print(f"\nUnique Categories: {unique_categories_count}")

# Monthly spending
expenses["month"] = expenses["booking_date"].dt.to_period("M").astype(str)
monthly_spending = (
    expenses.groupby("month")["amount"]
    .sum()
    .mul(-1)  # make positive
    .reset_index(name="amount_spent")
    .sort_values("month")
)

print("\nSpent by month")
print(monthly_spending)

# Spending by category
if "category" in expenses.columns:
    spending_by_category = (
        expenses.groupby("category")["amount"]
        .sum()
        .mul(-1)  # make positive
        .reset_index(name="amount_spent")
        .sort_values("amount_spent", ascending=False)
    )
    print("\nSpending by Category:")
    print(spending_by_category)
else:
    print("\nSpending by Category: (no 'category' column found)")



Total Spending: 38816.69 PLN

Unique Categories: 10

Spent by month
     month  amount_spent
0  2025-11       8902.82
1  2025-12      12870.01
2  2026-01       9630.50
3  2026-02       7413.36

Spending by Category:
        category  amount_spent
0     Accounting      10914.48
1      Apartment       9250.00
9  Uncategorized       8030.08
5   Intermediary       3365.90
4      Groceries       3329.24
6       Services       1815.00
3           Food       1277.65
8      Transport        382.14
2          Cofee        263.50
7  Subscriptions        188.70
